### Import libraries

In [1]:
from snowflake.snowpark.functions import col,trim,split,lit
from snowflake.snowpark.functions import col, sum as _sum, when, is_null
from snowflake.snowpark import functions as F
import sys 
sys.path.append(r"C:\Users\G0004878\Desktop\TFT_Data\utils_files")
import snowflake_utils
import Snowflake_configuration
snowflake_conn_prop = Snowflake_configuration.ds1_role_json
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import datetime
import math 
from dateutil.relativedelta import relativedelta
from snowflake_utils import *
import shutil
import os, json, glob, gc
import collections.abc
from snowflake.snowpark.types import StringType
from darts import TimeSeries
from darts.dataprocessing.transformers import (
    Scaler,
    StaticCovariatesTransformer,
)
from darts.models import TFTModel

The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Snowflake Session

In [2]:
session = Session.builder.configs(snowflake_conn_prop).create()
session.use_database('MOP_DATABASE')
session.use_schema('SOQ')

### Read the table

In [3]:
##Reading the table with all the series
df = session.table('MOP_DATABASE.SOQ.DAILY_DATA_FOR_FORECASTING_WITH_FESTIVE_FEATURES')

In [27]:
df.limit(5).to_pandas().to_csv(r"Sample_data_top_5_rows.csv")

### Separating columns by data types

In [6]:
from snowflake.snowpark.types import (
    IntegerType, LongType, ShortType, ByteType, DecimalType, # Numbers
    BooleanType,                                             # Boolean
    StringType,                                              # String
    TimestampType, DateType, TimeType,                       # Datetime
    FloatType, DoubleType                                    # Float
)

In [16]:
datetime_columns = [field.name for field in df.schema.fields if isinstance(field.datatype,(TimestampType,DateType, TimeType))]
boolean_columns = [field.name for field in df.schema.fields if isinstance(field.datatype,BooleanType)]
categorical_columns = [field.name for field in df.schema.fields if isinstance(field.datatype,StringType)]
float_columns = [field.name for field in df.schema.fields if isinstance(field.datatype,(FloatType, DoubleType))]
number_columns = [field.name for field in df.schema.fields if isinstance(field.datatype,(IntegerType, LongType, ShortType, ByteType, DecimalType))]

In [17]:
categorical_columns

['PARENT_DEALER_CODE',
 'MODEL_FAMILY',
 'MODEL_FAMILY_CODE',
 'DAY_OF_THE_WEEK',
 'PARENT_DEALER_CODE_MODEL_FAMILY',
 'MODEL_NAME',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME',
 'DATASET_TYPE']

In [19]:
number_columns

['YEAR',
 'MONTH',
 'DAY_OF_THE_MONTH',
 'NET_SALES',
 'HARTALIK_TEEJ',
 'GANESH_CHATURTHI',
 'JANMASHTAMI',
 'VISHWAKARMA_PUJA',
 'KARWA_CHAUTH',
 'ONAM',
 '"N-16"',
 '"N-15"',
 '"N-14"',
 '"N-13"',
 '"N-12"',
 '"N-11"',
 '"N-10"',
 '"N-9"',
 '"N-8"',
 '"N-7"',
 '"N-6"',
 '"N-5"',
 '"N-4"',
 '"N-3"',
 '"N-2"',
 '"N-1"',
 'N',
 '"N+1"',
 '"N+2"',
 '"N+3"',
 '"N+4"',
 '"N+5"',
 '"N+6"',
 '"N+7"',
 '"N+8"',
 '"N+9"',
 '"N+10"',
 '"D-3"',
 '"D-2"',
 '"D-1"',
 'D',
 '"D+1"',
 '"D+2"',
 '"D+3"',
 '"D+4"',
 '"D+5"',
 '"D+6"',
 'HANUMAN_JAYANTI',
 'AKSHYA_TRITIYA',
 'BUDDHA_PURNIMA',
 'GANGA_DUSSEHRA',
 'JAGANNATH_RATHYATRA',
 'GURU_PURNIMA',
 'NAG_PANCHAMI',
 'RAKSHA_BANDHAN',
 'MARRIAGE_DAY']

In [22]:
snowflake_utils.shape_of_snowpark_df(df)[1] == (len(datetime_columns) + len(categorical_columns) + len(number_columns))

True

In [28]:
from snowflake.snowpark.functions import count_distinct, lit
from functools import reduce

categorical_cols = [
    'PARENT_DEALER_CODE', 'MODEL_FAMILY', 'MODEL_FAMILY_CODE',
    'DAY_OF_THE_WEEK', 'PARENT_DEALER_CODE_MODEL_FAMILY',
    'MODEL_NAME', 'BRAKE_TYPE', 'IGNITION_TYPE', 'WHEEL_TYPE',
    'COLOUR', 'DEALER_CITY', 'X_CITY_CATEGORY',
    'ZONAL_OFFICE_NAME', 'DATASET_TYPE'
]

dfs = [
    df.select(
        lit(col).alias('COLUMN_NAME'),
        count_distinct(col).alias('DISTINCT_COUNT')
    )
    for col in categorical_cols
]

result = reduce(lambda a, b: a.union_all(b), dfs)
result.show()

------------------------------------------------------
|"COLUMN_NAME"                    |"DISTINCT_COUNT"  |
------------------------------------------------------
|PARENT_DEALER_CODE               |1188              |
|MODEL_FAMILY                     |13                |
|MODEL_FAMILY_CODE                |206               |
|DAY_OF_THE_WEEK                  |7                 |
|PARENT_DEALER_CODE_MODEL_FAMILY  |115940            |
|MODEL_NAME                       |13                |
|BRAKE_TYPE                       |2                 |
|IGNITION_TYPE                    |2                 |
|WHEEL_TYPE                       |4                 |
|COLOUR                           |31                |
------------------------------------------------------

